In [ ]:
from pathlib import Path
import json
import numpy as np
from PIL import Image, ImageDraw
from IPython.display import display
from skimage.segmentation import felzenszwalb, mark_boundaries

MANIFESTO = Path(
    "/workspace/ProtoSAM-repro/reproducao/"
    "protocolo_polipos_20261011T143313814944Z/manifesto.json"
)
dados = json.loads(MANIFESTO.read_text(encoding="utf-8"))

# Inspeção visual: não inicia treinamento nem lê a máscara manual.
registro = dados["pool_adaptacao"][0]
with Image.open(registro["imagem"]) as im:
    rgb = np.asarray(
        im.convert("RGB").resize((256, 256), Image.Resampling.BILINEAR)
    )

PARAMETROS_INSPECAO = {
    "scale": 100,
    "sigma": 0.8,
    "min_size": 50,
}

segmentos = felzenszwalb(
    rgb.astype(np.float32) / 255.0,
    channel_axis=-1,
    **PARAMETROS_INSPECAO,
)

# Heurística provisória para excluir pixels quase pretos.
# Não é a anotação do pólipo nem uma reprodução validada do filtro oficial.
area_util = rgb.max(axis=-1) > 10

candidatos = []
for identificador in np.unique(segmentos):
    regiao = segmentos == identificador
    area = int(regiao.sum())
    fracao_util = float(area_util[regiao].mean())
    if area >= 100 and fracao_util >= 0.95:
        candidatos.append(int(identificador))

assert candidatos, "Nenhuma região elegível; precisamos inspecionar a imagem."

rng = np.random.default_rng(1234)
identificador = int(rng.choice(candidatos))
mascara_support = (segmentos == identificador) & area_util

# Mesma região, com transformação geométrica correspondente.
query_rgb = np.flip(rgb, axis=1).copy()
mascara_query = np.flip(mascara_support, axis=1).copy()

def contorno(imagem, mascara):
    resultado = mark_boundaries(
        imagem.astype(np.float32) / 255.0,
        mascara.astype(np.int32),
        color=(0, 1, 0),
        mode="thick",
    )
    return (np.clip(resultado, 0, 1) * 255).astype(np.uint8)

todos = mark_boundaries(
    rgb.astype(np.float32) / 255.0,
    segmentos,
    color=(1, 1, 0),
)
todos = (np.clip(todos, 0, 1) * 255).astype(np.uint8)

paineis = [
    ("Imagem", rgb),
    ("Superpixels", todos),
    ("Support: regiao escolhida", contorno(rgb, mascara_support)),
    ("Query: mesma regiao", contorno(query_rgb, mascara_query)),
]

painel = Image.new("RGB", (4 * 256, 286), "white")
desenho = ImageDraw.Draw(painel)

for coluna, (titulo, imagem) in enumerate(paineis):
    x = coluna * 256
    desenho.text((x + 5, 8), titulo, fill="black")
    painel.paste(Image.fromarray(imagem), (x, 30))

display(painel)

print("Imagem:", registro["nome"])
print("Parâmetros de inspeção:", PARAMETROS_INSPECAO)
print("Superpixels gerados:", len(np.unique(segmentos)))
print("Regiões elegíveis:", len(candidatos))
print("Região escolhida:", identificador)
print("Pixels da região:", int(mascara_support.sum()))
print("Nenhuma máscara manual foi usada.")

In [ ]:
import torch
import torch.nn.functional as F

def verificar_mascara_alp(nome, mascara):
    m = torch.from_numpy(
        mascara.astype("float32")
    )[None, None]

    # Mesmo tipo de redução usado nas máscaras pelo ALP.
    reduzida = F.interpolate(m, size=(32, 32), mode="nearest")
    cobertura_fg = F.avg_pool2d(reduzida, kernel_size=4)
    cobertura_bg = F.avg_pool2d(1 - reduzida, kernel_size=4)

    pixels_fg = int(reduzida.sum().item())
    locais_fg = int((cobertura_fg >= 0.95).sum().item())
    locais_bg = int((cobertura_bg >= 0.95).sum().item())

    print(f"\n{nome}")
    print("Pixels FG na grade 32×32:", pixels_fg)
    print("Células locais elegíveis FG:", locais_fg)
    print("Células locais elegíveis BG:", locais_bg)

    if pixels_fg == 0:
        print("A região desapareceu na redução.")
    elif locais_fg == 0:
        print("A região sobreviveu; o FG usaria o protótipo global.")
    else:
        print("A região permite protótipos locais de FG.")

verificar_mascara_alp("Support", mascara_support)
verificar_mascara_alp("Query", mascara_query)

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from scipy import ndimage as ndi
from PIL import Image, ImageDraw
from skimage.segmentation import felzenszwalb, mark_boundaries
from IPython.display import display

rng = np.random.default_rng(1234)
pool = dados["pool_adaptacao"]
indices = rng.choice(len(pool), size=6, replace=False)

def obter_area_util(rgb):
    nao_preto = rgb.max(axis=-1) > 10
    componentes, quantidade = ndi.label(nao_preto)
    if quantidade == 0:
        return np.zeros(nao_preto.shape, dtype=bool)

    tamanhos = np.bincount(componentes.ravel())
    tamanhos[0] = 0
    maior = componentes == tamanhos.argmax()
    maior = ndi.binary_fill_holes(maior)

    return ndi.binary_erosion(
        maior,
        structure=np.ones((3, 3), dtype=bool),
        iterations=3,
        border_value=0,
    )

def estatisticas_grade(mascara):
    m = torch.from_numpy(mascara.astype(np.float32))[None, None]
    m = F.interpolate(m, size=(32, 32), mode="nearest")
    fg = int(m.sum())
    locais_fg = int((F.avg_pool2d(m, 4) >= 0.95).sum())
    locais_bg = int((F.avg_pool2d(1 - m, 4) >= 0.95).sum())
    return fg, locais_fg, locais_bg

def desenhar_contorno(rgb, mascara):
    vis = mark_boundaries(
        rgb.astype(np.float32) / 255,
        mascara.astype(np.int32),
        color=(0, 1, 0),
        mode="thick",
    )
    return (np.clip(vis, 0, 1) * 255).astype(np.uint8)

painel = Image.new("RGB", (4 * 256, 6 * 286), "white")
desenho = ImageDraw.Draw(painel)
episodios_inspecao = []

for linha, indice in enumerate(indices):
    registro = pool[int(indice)]
    with Image.open(registro["imagem"]) as im:
        rgb = np.asarray(
            im.convert("RGB").resize(
                (256, 256), Image.Resampling.BILINEAR
            )
        )

    util = obter_area_util(rgb)
    segmentos = felzenszwalb(
        rgb.astype(np.float32) / 255,
        channel_axis=-1,
        **PARAMETROS_INSPECAO,
    )

    candidatos = []
    for identificador in np.unique(segmentos):
        regiao = segmentos == identificador
        if regiao.sum() < 100:
            continue
        if util[regiao].mean() < 0.95:
            continue

        mascara = regiao & util
        query_mask = np.flip(mascara, axis=1).copy()
        stats_support = estatisticas_grade(mascara)
        stats_query = estatisticas_grade(query_mask)

        # Permite FG global; exige FG sobrevivente e BG local
        # nos dois lados deste episódio demonstrativo.
        if (
            stats_support[0] > 0 and stats_support[2] > 0
            and stats_query[0] > 0 and stats_query[2] > 0
        ):
            candidatos.append(int(identificador))

    colunas = [
        ("Imagem", rgb),
        ("Area util", np.repeat(
            (util.astype(np.uint8) * 255)[..., None], 3, axis=2
        )),
    ]

    if candidatos:
        escolhido = int(rng.choice(candidatos))
        mascara = (segmentos == escolhido) & util
        query_rgb = np.flip(rgb, axis=1).copy()
        query_mask = np.flip(mascara, axis=1).copy()

        colunas += [
            ("Support", desenhar_contorno(rgb, mascara)),
            ("Query espelhada", desenhar_contorno(query_rgb, query_mask)),
        ]
        episodios_inspecao.append({
            "nome": registro["nome"],
            "rgb": rgb,
            "mascara_support": mascara,
            "query_rgb": query_rgb,
            "mascara_query": query_mask,
            "regiao": escolhido,
        })

        print(
            f"{registro['nome']}: candidatos={len(candidatos)} | "
            f"regiao={escolhido} | area={int(mascara.sum())} | "
            f"grade support={estatisticas_grade(mascara)} | "
            f"grade query={estatisticas_grade(query_mask)}"
        )
    else:
        vazio = np.zeros_like(rgb)
        colunas += [
            ("Sem regiao elegivel", vazio),
            ("Episodio nao criado", vazio),
        ]
        print(f"{registro['nome']}: nenhuma região elegível.")

    for coluna, (titulo, imagem) in enumerate(colunas):
        x, y = coluna * 256, linha * 286
        desenho.text((x + 5, y + 8), titulo, fill="black")
        painel.paste(Image.fromarray(imagem), (x, y + 30))

display(painel)
print("\nGrade: (pixels FG, células locais FG, células locais BG)")
print("Episódios criados:", len(episodios_inspecao), "/ 6")

In [ ]:
import json
import subprocess
from pathlib import Path

repo = Path("/workspace/ProtoSAM-repro")
commit = "64868c909668aac9ad517f7192a2712e88268e8f"

conteudo = subprocess.check_output(
    ["git", "show", f"{commit}:data/data_processing.ipynb"],
    cwd=repo,
    text=True,
)
notebook = json.loads(conteudo)

termos = ("felzenszwalb", "superpix", "fgmask", "polyp")
encontradas = []

for indice, celula in enumerate(notebook["cells"]):
    fonte = celula.get("source", [])
    texto = fonte if isinstance(fonte, str) else "".join(fonte)
    if any(termo in texto.lower() for termo in termos):
        encontradas.append((indice, texto))

print("Células relevantes:", len(encontradas))
for indice, texto in encontradas:
    print(f"\n{'=' * 60}\nCÉLULA {indice}\n{'=' * 60}")
    print(texto)

In [ ]:
import numpy as np
import pandas as pd
from skimage.segmentation import felzenszwalb

assert len(episodios_inspecao) == 6

configuracoes_sp = {
    "inspecao_atual": dict(scale=100, sigma=0.8, min_size=50),
    # Parâmetros inspirados no notebook CT/MRI.
    # Não representam uma configuração oficial confirmada para pólipos.
    "regioes_maiores": dict(scale=100, sigma=1.0, min_size=400),
}

comparacao_sp = []
mapas_sp_comparacao = {}

for episodio in episodios_inspecao:
    nome = episodio["nome"]
    rgb = episodio["rgb"]
    area_util = obter_area_util(rgb).astype(bool)

    for configuracao, parametros in configuracoes_sp.items():
        segmentos = felzenszwalb(
            rgb.astype(np.float32) / 255.0,
            channel_axis=-1,
            **parametros,
        )
        mapas_sp_comparacao[(nome, configuracao)] = segmentos

        areas = []
        for regiao in np.unique(segmentos):
            pixels = segmentos == regiao
            area_total = int(pixels.sum())
            area_interna = int((pixels & area_util).sum())

            # Mesmo filtro da inspeção anterior.
            if area_interna >= 100 and area_interna / area_total >= 0.95:
                areas.append(area_interna)

        comparacao_sp.append({
            "imagem": nome,
            "configuracao": configuracao,
            "regioes_totais": int(np.unique(segmentos).size),
            "regioes_elegiveis": len(areas),
            "area_minima": min(areas) if areas else np.nan,
            "area_mediana": float(np.median(areas)) if areas else np.nan,
            "area_maxima": max(areas) if areas else np.nan,
        })

comparacao_sp = pd.DataFrame(comparacao_sp)
display(comparacao_sp)

print("Áreas medidas em pixels na imagem 256×256.")
print("Episódios anteriores preservados; nenhum treinamento executado.")

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

CONFIG_SP_EFT = {
    "scale": 100,
    "sigma": 1.0,
    "min_size": 400,
}

def verificar_grade_eft(mascara):
    tensor = torch.from_numpy(
        np.ascontiguousarray(mascara)
    ).float()[None, None]

    grade = F.interpolate(tensor, size=(32, 32), mode="nearest")
    fg_local = F.avg_pool2d(grade, kernel_size=4, stride=4)
    bg_local = F.avg_pool2d(1 - grade, kernel_size=4, stride=4)

    return {
        "pixels_fg": int(grade.sum().item()),
        "celulas_fg": int((fg_local > 0.95).sum().item()),
        "celulas_bg": int((bg_local > 0.95).sum().item()),
    }

rng_eft = np.random.default_rng(1234)
episodios_eft = []

for anterior in episodios_inspecao:
    nome = anterior["nome"]
    rgb = anterior["rgb"]
    area_util = obter_area_util(rgb).astype(bool)
    segmentos = mapas_sp_comparacao[(nome, "regioes_maiores")]

    candidatos = []

    for regiao in np.unique(segmentos):
        regiao_original = segmentos == regiao
        mascara = regiao_original & area_util

        if mascara.sum() < 100:
            continue
        if mascara.sum() / regiao_original.sum() < 0.95:
            continue

        mascara_query = np.fliplr(mascara).copy()
        grade_support = verificar_grade_eft(mascara)
        grade_query = verificar_grade_eft(mascara_query)

        # FG global é permitido; não exigimos células locais FG.
        if all(
            g["pixels_fg"] > 0 and g["celulas_bg"] > 0
            for g in (grade_support, grade_query)
        ):
            candidatos.append((
                int(regiao), mascara, mascara_query,
                grade_support, grade_query,
            ))

    assert candidatos, f"Sem episódio válido para {nome}"

    escolhido = candidatos[int(rng_eft.integers(len(candidatos)))]
    regiao, mascara, mascara_query, gs, gq = escolhido

    episodios_eft.append({
        "nome": nome,
        "rgb": rgb.copy(),
        "mascara_support": mascara.copy(),
        "query_rgb": np.fliplr(rgb).copy(),
        "mascara_query": mascara_query,
        "regiao": regiao,
    })

    print(
        f"{nome} | candidatos={len(candidatos)} "
        f"| área={int(mascara.sum())} "
        f"| grade support={gs} | grade query={gq}"
    )

print(f"\nEpisódios preparados: {len(episodios_eft)} / 6")
print("Nenhuma máscara manual utilizada.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np

destino_episodios = (
    Path("/workspace/ProtoSAM-repro/reproducao")
    / (
        "eft_sp_episodios_inspecao_"
        + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    )
)
destino_episodios.mkdir(parents=True, exist_ok=False)

registros = []

for i, episodio in enumerate(episodios_eft):
    arquivo = f"episodio_{i:02d}.npz"

    np.savez_compressed(
        destino_episodios / arquivo,
        support_rgb=episodio["rgb"],
        support_mask=episodio["mascara_support"].astype(np.uint8),
        query_rgb=episodio["query_rgb"],
        query_mask=episodio["mascara_query"].astype(np.uint8),
    )

    registros.append({
        "arquivo": arquivo,
        "imagem": episodio["nome"],
        "regiao": int(episodio["regiao"]),
    })

config_inspecao = {
    "finalidade": "Teste de forward, perda e backward; não é treino completo",
    "metodo": "EFT-SP experimental para pólipos",
    "superpixels": CONFIG_SP_EFT,
    "resolucao": [256, 256],
    "transformacao_query": "flip horizontal da imagem e da mesma máscara",
    "semente_selecao_regiao": 1234,
    "mascaras_manuais_utilizadas": False,
    "episodios": registros,
}

(destino_episodios / "config.json").write_text(
    json.dumps(config_inspecao, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Episódios salvos em:", destino_episodios)